In [8]:
%pip install requests python-dotenv

import json
import os
from dotenv import load_dotenv

load_dotenv()


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [10]:
# config/kimi_client.py
import os
from openai import OpenAI

# Initialize Kimi K2 client
client = OpenAI(
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url="https://api.moonshot.ai/v1"
)

# Available models
KIMI_MODELS = {
    "k2-thinking": "kimi-k2-thinking",  # Best for reasoning
    "k2-preview": "moonshot/kimi-k2-preview",  # Latest preview
    "k2-0905": "moonshot/kimi-k2-0905-preview"  # September version
}

# Base costs (per million tokens)
COSTS = {
    "k2-thinking": {"input": 0.15, "output": 2.50}
}

In [17]:
# kim_modes/researcher.py
import os
import json
from openai import OpenAI

# Initialize Kimi client
client = OpenAI(
    api_key=os.getenv("MOONSHOT_API_KEY", "your-key-here"),
    base_url="https://api.moonshot.ai/v1"
)

# Define search function (use SerpAPI or mock for now) as mock function
def execute_search(query: str) -> str:
    """
    Implement actual search here (SerpAPI, Google API, etc.)
    For MVP, return mock results
    """
    return f"Mock search results for '{query}': Recent 2024 breakthroughs in quantum computing include IBM's 1000+ qubit processor, Google's quantum error correction milestone, and China's quantum communication satellite advances."

def researcher_mode(query: str, enable_search: bool = True):
    """
    Kimi K2 with function calling for search
    """
    # CORRECT tool definition - use "function" type
    tools = [{
        "type": "function",
        "function": {
            "name": "execute_search",
            "description": "Search the web for current information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }] if enable_search else None

    messages = [{"role": "user", "content": query}]
    
    # First call: Kimi decides if it needs search
    response = client.chat.completions.create(
        model="kimi-k2-thinking",
        messages=messages,
        tools=tools,
        temperature=0.3
    )
    
    message = response.choices[0].message
    
    # Check if Kimi requested a tool call
    if message.tool_calls:
        # Add assistant's tool request to messages
        messages.append({
            "role": message.role,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in message.tool_calls
            ]
        })
        
        # Execute the search function
        for tool_call in message.tool_calls:
            function_args = json.loads(tool_call.function.arguments)
            search_results = execute_search(function_args["query"])
            
            # Add tool results to messages
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": search_results
            })
        
        # Second call: Kimi processes search results and gives final answer
        final_response = client.chat.completions.create(
            model="kimi-k2-thinking",
            messages=messages,
            temperature=0.3
        )
        
        return final_response.choices[0].message.content
    
    # No tool needed - direct response
    return message.content

# Test it
print("=" * 70)
result = researcher_mode(
    "What are the latest breakthroughs in quantum computing in 2024?",
    enable_search=True
)
print("\n✅ Final Result:", result[:800])


✅ Final Result: Here are the major quantum computing breakthroughs and developments in 2024:

## **1. IBM's 1,000+ Qubit Milestone**
IBM unveiled its **Flamingo processor** with 1,121 superconducting qubits, marking the first time a quantum processor has crossed the 1,000-qubit threshold for public cloud access. More importantly, IBM demonstrated improved error rates and coherence times, focusing on quality over sheer quantity. They've also advanced their **Quantum System Two** architecture, enabling modular scaling.

## **2. Google's Quantum Error Correction Breakthrough**
Google achieved a critical milestone in **quantum error correction** by demonstrating that logical qubit error rates can be suppressed below physical qubit error rates. Using their **Willow chip** (105 physical qubits), they showed tha
